# Audit Hypothesis Agent — v2

RAG-агент для генерации аудиторских гипотез.

## Что исправлено по сравнению с v1:
- **Bug fix**: опечатка `normalize_embeddigns` → `normalize_embeddings` (запрос теперь нормализуется корректно)
- **Bug fix**: `add_generation_promt` → `add_generation_prompt` (chat-template работает правильно)
- **Bug fix**: дедупликация в `retrieve_with_rerank` перенесена за пределы цикла
- **Bug fix**: `dict_values` → `list` для корректной работы `zip`
- **Bug fix**: `rerunker` → `reranker` (единообразное именование)
- **Bug fix**: вложенные кавычки в f-string заменены на `'`
- **Feature**: добавлена реальная multi-query экспансия через LLM
- **Feature**: загрузка PDF и DOCX документов
- **Feature**: управление длиной контекста по токенам
- **Feature**: структурированный парсинг гипотез (Pydantic)
- **Refactor**: класс `AuditAgent` инкапсулирует логику вместо глобальных переменных
- **Refactor**: `DocumentLoader` — отдельный класс для загрузки документов
- **Security**: JSON-сериализация чанков вместо небезопасного pickle

## 1. Установка зависимостей

In [1]:
%pip install -U sentence-transformers
%pip install python-docx mammoth pdfplumber
%pip install -U bitsandbytes
%pip install langchain_text_splitters faiss-cpu
%pip install pydantic

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%pip install -q pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Юлька\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1.5 Конфигурация моделей

Редактируй `config.yaml` — ноутбук менять не нужно:
- Если папка `local_path` существует → используется локальная копия
- Если нет → модель скачивается с HuggingFace Hub автоматически
- `force_hub: true` → всегда скачивать (полезно для обновлений)

In [7]:
import os
import yaml


def resolve_model_path(local_path: str, hub_id: str, force_hub: bool = False) -> str:
    """
    Возвращает путь к модели:
      - локальный путь, если папка существует и force_hub=False
      - Hub ID в остальных случаях (from_pretrained скачает автоматически)
    """
    if not force_hub and os.path.isdir(local_path):
        print(f"✓ Локальная модель найдена: {local_path}")
        return local_path
    if not force_hub:
        print(f"⚠ Папка '{local_path}' не найдена.")
    print(f"⬇ Используем HuggingFace Hub: {hub_id}")
    return hub_id


# Загружаем конфигурацию
with open("config.yaml", "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)["models"]

model_path    = resolve_model_path(**_cfg["llm"])
embedder_path = resolve_model_path(**_cfg["embedder"])
reranker_path = resolve_model_path(**_cfg["reranker"])

print(f"\nLLM      : {model_path}")
print(f"Embedder : {embedder_path}")
print(f"Reranker : {reranker_path}")

⚠ Папка 'models/Qwen_Qwen2.5-7B-Instruct' не найдена.
⬇ Используем HuggingFace Hub: Qwen/Qwen2.5-7B-Instruct
⚠ Папка 'models/BAAI_bge-m3' не найдена.
⬇ Используем HuggingFace Hub: BAAI/bge-m3
⚠ Папка 'models/BAAI_bge-reranker-v2-m3' не найдена.
⬇ Используем HuggingFace Hub: BAAI/bge-reranker-v2-m3

LLM      : Qwen/Qwen2.5-7B-Instruct
Embedder : BAAI/bge-m3
Reranker : BAAI/bge-reranker-v2-m3


## 2. Загрузка моделей

In [8]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16"
)

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

print(f"Занято в памяти: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Загрузка моделей...")

# Пути берутся из config.yaml (ячейка выше)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cpu",
    quantization_config=quant_config,
)
print(model.device)
print(f"Модель загружена: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

embedder = SentenceTransformer(embedder_path, device="cpu")
reranker = CrossEncoder(reranker_path, device="cpu")

print(f"Reranker загружен: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Модели загружены")

Занято в памяти: 0.00 GB
Загрузка моделей...


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

## 3. Загрузчик документов

Новый класс — отдельная ответственность для парсинга PDF и DOCX.

In [ ]:
import mammoth
import pdfplumber
import os
from typing import Optional


class DocumentLoader:
    """Загружает текст из PDF, DOCX и TXT файлов."""

    @staticmethod
    def load(file_path: str) -> str:
        """Определяет тип файла и вызывает нужный парсер."""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Файл не найден: {file_path}")

        ext = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            return DocumentLoader._load_pdf(file_path)
        elif ext in (".docx", ".doc"):
            return DocumentLoader._load_docx(file_path)
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read()
        else:
            raise ValueError(f"Неподдерживаемый формат: {ext}")

    @staticmethod
    def _load_pdf(file_path: str) -> str:
        """Извлекает текст из PDF."""
        pages = []
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages.append(text)
        result = "\n\n".join(pages)
        print(f"PDF загружен: {len(pages)} страниц, {len(result)} символов")
        return result

    @staticmethod
    def _load_docx(file_path: str) -> str:
        """Извлекает текст из DOCX через mammoth."""
        with open(file_path, "rb") as f:
            result = mammoth.extract_raw_text(f)
        text = result.value
        print(f"DOCX загружен: {len(text)} символов")
        return text

## 4. Ретривер

**Изменения vs v1:**
- `normalize_embeddings` (опечатка исправлена)
- Хранение через JSON вместо pickle (безопасность)
- Опциональный HNSW-индекс для больших коллекций
- `add_document` принимает путь к файлу напрямую

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from typing import List, Dict
import faiss
import json


class DocumentRetriever:
    def __init__(self, embedder, chunk_size: int = 800, chunk_overlap: int = 100,
                 index_dir: str = ".", use_hnsw: bool = False):
        """
        Args:
            embedder: SentenceTransformer-совместимый энкодер.
            chunk_size: Размер чанка в символах.
            chunk_overlap: Перекрытие чанков.
            index_dir: Директория для сохранения индекса.
            use_hnsw: Использовать HNSW (быстрее для >50k векторов).
        """
        self.embedder = embedder
        self.chunks: List[Dict] = []
        self.index = None
        self.dimension: Optional[int] = None
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.use_hnsw = use_hnsw

        self._index_path = os.path.join(index_dir, "index.faiss")
        self._chunks_path = os.path.join(index_dir, "chunks.json")

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", ", ", " ", ""],
            length_function=len,
            is_separator_regex=False,
        )
        self._load_existing_db()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def add_file(self, file_path: str):
        """Загружает файл и добавляет его в БД."""
        text = DocumentLoader.load(file_path)
        metadata = {"source": os.path.basename(file_path)}
        self.add_document(text, metadata)

    def add_document(self, text: str, metadata: Optional[Dict] = None):
        """Добавляет текст в БД, эмбеддинги строятся только для нового документа."""
        print("Разбиваем текст на чанки...")
        new_chunks = self.text_splitter.split_text(text)

        if not new_chunks:
            print("Нет чанков для добавления")
            return

        print(f"Создание эмбеддингов для {len(new_chunks)} новых чанков...")
        new_embeddings = self.embedder.encode(
            new_chunks,
            batch_size=32,
            normalize_embeddings=True,   # FIX: было normalize_embeddigns
            show_progress_bar=True,
        ).astype("float32")

        if self.index is None:
            self.dimension = new_embeddings.shape[1]
            self.index = self._create_index(self.dimension)
            print(f"Создан новый FAISS индекс (dim={self.dimension}, hnsw={self.use_hnsw})")

        self.index.add(new_embeddings)

        for chunk in new_chunks:
            self.chunks.append({"text": chunk, "metadata": metadata or {}})

        print(f"Добавлено {len(new_chunks)} чанков. Всего в БД: {len(self.chunks)}")
        self._save_db()

    def search(self, query: str, top_k: int = 10) -> List[Dict]:
        """Ищет top_k релевантных чанков по запросу."""
        if self.index is None or self.index.ntotal == 0:
            print("БД пуста. Добавьте документы через add_document() или add_file().")
            return []

        query_embedding = self.embedder.encode(
            [query],
            normalize_embeddings=True,   # FIX: было normalize_embeddigns
        ).astype("float32")

        k = min(top_k, self.index.ntotal)
        scores, indices = self.index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1 and idx < len(self.chunks):
                results.append({
                    "text": self.chunks[idx]["text"],
                    "metadata": self.chunks[idx]["metadata"],
                    "similarity": float(score),
                })
        return results

    def get_stats(self) -> Dict:
        return {
            "total_chunks": len(self.chunks),
            "total_vectors": self.index.ntotal if self.index else 0,
            "dimension": self.dimension,
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "index_type": "HNSW" if self.use_hnsw else "FlatIP",
        }

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _create_index(self, dim: int):
        if self.use_hnsw:
            # HNSW: O(log n) поиск, подходит для >50k векторов
            index = faiss.IndexHNSWFlat(dim, 32)
            index.hnsw.efConstruction = 200
            index.hnsw.efSearch = 64
            return index
        else:
            # FlatIP: точный cosine-поиск, подходит для <100k векторов
            return faiss.IndexFlatIP(dim)

    def _load_existing_db(self):
        """Загружает сохранённую БД."""
        if not os.path.exists(self._index_path) or not os.path.exists(self._chunks_path):
            print("Сохранённая БД не найдена. Будет создана новая.")
            return False
        try:
            self.index = faiss.read_index(self._index_path)
            self.dimension = self.index.d
            with open(self._chunks_path, "r", encoding="utf-8") as f:
                data = json.load(f)   # FIX: JSON вместо pickle (безопасность)
                self.chunks = data["chunks"]
                self.chunk_size = data.get("chunk_size", self.chunk_size)
                self.chunk_overlap = data.get("chunk_overlap", self.chunk_overlap)
            print(f"БД загружена: {self.index.ntotal} векторов, {len(self.chunks)} чанков")
            return True
        except Exception as e:
            print(f"Ошибка загрузки БД: {e}")
            return False

    def _save_db(self):
        """Сохраняет БД на диск."""
        if self.index is None or self.index.ntotal == 0:
            return
        faiss.write_index(self.index, self._index_path)
        with open(self._chunks_path, "w", encoding="utf-8") as f:
            json.dump(
                {"chunks": self.chunks, "chunk_size": self.chunk_size, "chunk_overlap": self.chunk_overlap},
                f, ensure_ascii=False, indent=2,
            )
        print(f"БД сохранена: {self.index.ntotal} векторов, {len(self.chunks)} чанков")

## 5. Multi-query + Reranking

**Изменения vs v1:**
- `expand_query` теперь реально генерирует альтернативные формулировки через LLM
- дедупликация вынесена за пределы цикла
- `dict_values` явно конвертируется в `list` перед `zip`

In [ ]:
def expand_query(question: str, model, tokenizer, n_variants: int = 3) -> List[str]:
    """
    Генерирует n_variants перефразировок вопроса для multi-query retrieval.
    Улучшает recall, особенно для коротких запросов.
    """
    prompt = (
        f"Сгенерируй {n_variants} разных формулировки следующего аудиторского вопроса. "
        f"Каждую формулировку напиши на новой строке, без нумерации и пояснений.\n\n"
        f"Вопрос: {question}\n\nФормулировки:"
    )
    messages = [
        {"role": "system", "content": "Ты — помощник аудитора. Перефразируй запросы точно и кратко."},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # FIX: было add_generation_promt
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,    # низкая температура для более детерминированных перефразировок
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    variants = [line.strip() for line in raw.splitlines() if line.strip()]
    # Всегда включаем оригинальный вопрос первым
    queries = [question] + variants[:n_variants]
    print(f"Multi-query: {len(queries)} формулировок")
    return queries


def retrieve_with_rerank(question: str, retriever: DocumentRetriever,
                         reranker: CrossEncoder,
                         model=None, tokenizer=None,
                         top_k: int = 8,
                         use_multi_query: bool = True) -> List[Dict]:
    """
    1. (опционально) расширяет запрос через LLM
    2. ищет по всем формулировкам
    3. дедуплицирует
    4. ранжирует через CrossEncoder
    """
    if use_multi_query and model is not None and tokenizer is not None:
        queries = expand_query(question, model, tokenizer)
    else:
        queries = [question]

    # Сбор результатов по всем запросам
    all_results = []
    for q in queries:
        all_results.extend(retriever.search(q, top_k=top_k))

    # FIX: дедупликация вынесена за пределы цикла
    unique_map = {r["text"]: r for r in all_results}
    unique_list = list(unique_map.values())   # FIX: явный list() для корректной работы zip

    if not unique_list:
        return []

    texts = [r["text"] for r in unique_list]
    pairs = [[question, t] for t in texts]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(unique_list, scores),   # FIX: list уже материализован
        key=lambda x: x[1],
        reverse=True,
    )
    return [r for r, _score in ranked[:top_k]]

## 6. Структурированный вывод (Pydantic)

Парсинг ответа LLM в типизированные объекты — упрощает downstream-обработку.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
import re


class AuditHypothesis(BaseModel):
    """Одна аудиторская гипотеза."""
    number: int
    title: str = Field(description="Краткое описание гипотезы")
    rationale: str = Field(description="Обоснование")
    consequences: str = Field(description="Возможные последствия")
    verification_steps: str = Field(description="Шаги проверки")


class HypothesesReport(BaseModel):
    """Полный отчёт с гипотезами."""
    question: str
    hypotheses: List[AuditHypothesis]
    sources_used: int

    def to_markdown(self) -> str:
        """Форматирует отчёт в читаемый Markdown."""
        lines = [f"## Аудиторские гипотезы\n", f"**Вопрос:** {self.question}\n"]
        for h in self.hypotheses:
            lines.append(f"### Гипотеза {h.number}: {h.title}")
            lines.append(f"- **Обоснование:** {h.rationale}")
            lines.append(f"- **Последствия:** {h.consequences}")
            lines.append(f"- **Проверка:** {h.verification_steps}\n")
        lines.append(f"*Использовано источников: {self.sources_used}*")
        return "\n".join(lines)


def parse_hypotheses(raw_answer: str, question: str, sources_count: int) -> HypothesesReport:
    """
    Парсит сырой текст LLM в структурированный HypothesesReport.
    Использует регулярные выражения для извлечения секций.
    """
    hypotheses = []
    # Ищем блоки вида "1. Гипотеза 1: ..."
    blocks = re.split(r"\n(?=\d+\.\s+(?:Гипотеза|Hypothesis))", raw_answer.strip())

    for i, block in enumerate(blocks, 1):
        if not block.strip():
            continue
        title_match = re.search(r"^\d+\.\s+(?:Гипотеза \d+:\s*)?(.+)", block, re.MULTILINE)
        title = title_match.group(1).strip() if title_match else f"Гипотеза {i}"

        rationale = _extract_section(block, ["Обоснование", "Обоснование:"])
        consequences = _extract_section(block, ["Последствия", "Последствия:"])
        verification = _extract_section(block, ["Проверка", "Проверка:", "Шаги проверки"])

        hypotheses.append(AuditHypothesis(
            number=i,
            title=title,
            rationale=rationale or "—",
            consequences=consequences or "—",
            verification_steps=verification or "—",
        ))

    return HypothesesReport(
        question=question,
        hypotheses=hypotheses,
        sources_used=sources_count,
    )


def _extract_section(text: str, labels: List[str]) -> Optional[str]:
    """Извлекает содержимое секции по одному из меток-заголовков."""
    for label in labels:
        pattern = rf"-?\s*\*?\*?{re.escape(label)}\*?\*?:?\s*(.+?)(?=\n\s*-|\n\s*\*|$)"
        m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None

## 7. Основной агент

Класс `AuditAgent` инкапсулирует все компоненты вместо глобальных переменных.

In [ ]:
class AuditAgent:
    """
    RAG-агент для генерации аудиторских гипотез.

    Инкапсулирует retriever, model, tokenizer, reranker.
    """

    # Максимальная длина контекста в символах (~75% от лимита модели)
    MAX_CONTEXT_CHARS = 12_000

    def __init__(self, model, tokenizer, embedder, reranker,
                 chunk_size: int = 800, chunk_overlap: int = 100,
                 index_dir: str = "."):
        self.model = model
        self.tokenizer = tokenizer
        self.reranker = reranker
        self.retriever = DocumentRetriever(
            embedder=embedder,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            index_dir=index_dir,
        )

    def add_file(self, file_path: str):
        """Добавляет документ в базу знаний."""
        self.retriever.add_file(file_path)

    def ask(self, question: str, top_k: int = 8,
            use_multi_query: bool = True) -> HypothesesReport:
        """
        Главный метод: принимает вопрос, возвращает структурированный отчёт.

        Args:
            question: Аудиторский вопрос на русском языке.
            top_k: Количество чанков для контекста.
            use_multi_query: Включить расширение запроса через LLM.
        """
        print("[1/3] Поиск релевантных фрагментов...")
        chunks = retrieve_with_rerank(
            question, self.retriever, self.reranker,
            model=self.model if use_multi_query else None,
            tokenizer=self.tokenizer if use_multi_query else None,
            top_k=top_k,
            use_multi_query=use_multi_query,
        )

        if not chunks:
            print("Предупреждение: база знаний пуста. Гипотезы будут сгенерированы без контекста.")

        context = self._build_context(chunks)
        prompt = self._build_prompt(context, question)

        print("[2/3] Генерация гипотез...")
        raw_answer = self._generate(prompt)

        print("[3/3] Парсинг результатов...")
        report = parse_hypotheses(raw_answer, question, len(chunks))

        return report

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build_context(self, chunks: List[Dict]) -> str:
        """Строит контекст с ограничением по длине."""
        parts = []
        total = 0
        for i, chunk in enumerate(chunks):
            part = f"[ИСТОЧНИК {i + 1}]\n{chunk['text']}"
            if total + len(part) > self.MAX_CONTEXT_CHARS:
                print(f"Контекст обрезан до {i} чанков (лимит {self.MAX_CONTEXT_CHARS} символов)")
                break
            parts.append(part)
            total += len(part)
        return "\n\n---\n\n".join(parts)

    def _build_prompt(self, context: str, question: str) -> str:
        return f"""Ты — AI-агент для генерации аудиторских гипотез. На основе предоставленных документов предложи 3-5 обоснованных гипотез для дальнейшего аудиторского анализа.

Каждая гипотеза должна содержать:
- краткое описание сути;
- обоснование (почему актуальна, на какие риски указывает);
- возможные последствия;
- шаги для проверки.

Формат ответа:
1. Гипотеза 1: [описание]
   - Обоснование: [аргументы]
   - Последствия: [кратко]
   - Проверка: [шаги]
2. Гипотеза 2:
   ...

Критерии: гипотезы должны быть конкретными, релевантными и проверяемыми.

Контекст для анализа:
{context}

Вопрос:
{question}
"""

    def _generate(self, prompt: str) -> str:
        """Генерирует ответ через LLM."""
        messages = [
            {"role": "system", "content": "Ты — ассистент ведущего аудитора банковской группы."},
            {"role": "user", "content": prompt},
        ]
        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,   # FIX: было add_generation_promt
        )
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.6,   # чуть ниже для более детерминированных гипотез
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(generated_ids, skip_special_tokens=True)

## 8. Инициализация агента

In [ ]:
agent = AuditAgent(
    model=model,
    tokenizer=tokenizer,
    embedder=embedder,
    reranker=reranker,
    chunk_size=800,
    chunk_overlap=100,
    index_dir=".",
)

print("Агент готов.")
print("Статистика базы знаний:", agent.retriever.get_stats())

## 9. Загрузка документов (пример)

In [ ]:
# Пример загрузки документов:
# agent.add_file("documents/policy_cards.pdf")
# agent.add_file("documents/transactions_2024.docx")
# agent.add_file("documents/internal_rules.txt")

print("Загрузите документы через agent.add_file('путь/к/файлу')")

## 10. Запрос к агенту

In [ ]:
question = (
    "Вычитка информации о финансовых транзакциях по бизнес-карте "
    "корпоративного клиента и их отображение по счёту клиента"
)

report = agent.ask(question, top_k=8, use_multi_query=True)

# Структурированный вывод
print("\n" + "=" * 60)
print(report.to_markdown())

print("\n" + "=" * 60)
print(f"Сгенерировано гипотез: {len(report.hypotheses)}")
print(f"Использовано источников: {report.sources_used}")